# Historical 50k pre-linkage assessment

This notebook is an interactive, Splink-native version of the first part
of the pre-linkage report. It examines whether identifiers are present and
distinguishing **before** fitting a linkage model.

The example uses one complete dataset because the task is deduplication.
Splink calls it the historical 50k dataset, although it contains 50,578
rows.

## 1. Choose what to assess

Edit these lists to test different identifiers and progressively relaxed
exact combinations. `BLANK_AS_NULL=True` treats empty and whitespace-only
strings as unavailable.

In [1]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

working_directory = Path.cwd().resolve()
candidates = [
    working_directory,
    working_directory / "splink_prelinkage",
    *working_directory.parents,
]
project_root = next(
    (
        path
        for path in candidates
        if (path / "splink").is_dir() and (path / "pyproject.toml").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError(
        "Could not locate the splink_prelinkage repository. Start Jupyter "
        "from that repository or its parent directory."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reload the local checkout when this notebook is rerun after source edits.
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "splink" or module_name.startswith("splink."):
        del sys.modules[module_name]

from splink import DuckDBAPI, splink_datasets
from splink.exploratory import (
    build_pre_linkage_profile,
    combination_ambiguity_chart,
    combination_ambiguity_data,
    combination_size_distribution_chart,
    dataset_linkage_overview_chart,
    dataset_linkage_overview_data,
    exact_duplicate_chart,
    exact_duplicate_data,
    linkage_column_quality_chart,
    linkage_column_quality_data,
    missingness_by_column_chart,
    missingness_by_column_data,
    missingness_by_count_data,
    missingness_pattern_chart,
    missingness_pattern_data,
    pre_linkage_reference_comparison_chart,
    reference_value_concentration_chart,
    reference_value_dominance_distribution_chart,
    subgroup_combination_ambiguity_chart,
    subgroup_combination_ambiguity_data,
    subgroup_equity_gap_chart,
    subgroup_equity_gap_data,
    subgroup_linkage_column_quality_chart,
    subgroup_linkage_column_quality_data,
    subgroup_missingness_chart,
    subgroup_missingness_summary_data,
    subgroup_value_concentration_chart,
    subgroup_value_dominance_distribution_chart,
    value_concentration_chart,
    value_dominance_distribution_chart,
)

LINKAGE_COLUMNS = [
    "first_name",
    "surname",
    "dob",
    "postcode_fake",
    "birth_place",
    "occupation",
]
COLUMN_COMBINATIONS = [
    ["first_name", "surname", "dob", "postcode_fake"],
    ["surname", "dob", "postcode_fake"],
    ["first_name", "dob", "postcode_fake"],
    ["dob", "postcode_fake"],
    ["surname", "dob"],
    ["postcode_fake"],
]
SUBGROUP_COMBINATIONS = [
    ["first_name", "surname", "dob", "postcode_fake"],
    ["dob", "postcode_fake"],
]
DISTRIBUTION_COLUMNS = ["first_name", "surname", "postcode_fake"]
AUXILIARY_COLUMNS = ["gender"]
DUPLICATE_COLUMNS = [
    "first_name",
    "surname",
    "dob",
    "postcode_fake",
    "birth_place",
    "occupation",
    "gender",
]
RECORD_ID_COLUMN = "unique_id"
# Set this to a JSON profile created by a previous approved assessment.
REFERENCE_PROFILE_PATH = None
BLANK_AS_NULL = True

## 2. Load the data locally

The identifiers stay in DuckDB. The assessment outputs contain column
names and aggregate counts, but not observed names, dates, postcodes or
identifier combinations.

In [2]:
db_api = DuckDBAPI()
historical_50k = splink_datasets.historical_50k
records = db_api.register(historical_50k)

pd.DataFrame(
    {
        "measure": ["Rows", "Columns", "Assessment purpose"],
        "value": [
            f"{historical_50k.num_rows:,}",
            str(len(historical_50k.column_names)),
            "Pre-linkage assessment for deduplication",
        ],
    }
)

,measure,value
0,Rows,"50,578"
1,Columns,11
2,Assessment purpose,Pre-linkage assessment for deduplication


## 3. Declared schema and roles

This declaration tells the reader how fields are used. It does not alter
the source data. `cluster` is a known duplicate-group label retained for
later evaluation; it is not used in this assessment.

In [3]:
roles = {
    "unique_id": ("record identifier", "Identifies a submitted row."),
    "cluster": ("evaluation label", "Known duplicate group; not used here."),
    "first_name": ("linkage column", "Candidate personal identifier."),
    "surname": ("linkage column", "Candidate personal identifier."),
    "dob": ("linkage column", "Candidate date-of-birth identifier."),
    "postcode_fake": ("linkage column", "Synthetic postcode-like identifier."),
    "birth_place": ("linkage column", "Candidate place identifier."),
    "occupation": ("linkage column", "Candidate contextual identifier."),
    "gender": ("auxiliary variable", "Available for subgroup assessment."),
    "full_name": ("derived identifier", "Combined name representation."),
    "first_and_surname": ("derived identifier", "Combined name representation."),
}
schema_rows = []
for field, data_type in zip(historical_50k.column_names, historical_50k.schema.types):
    role, meaning = roles.get(field, ("other", "Not declared for this example."))
    schema_rows.append(
        {
            "field": field,
            "data_type": str(data_type),
            "declared_role": role,
            "meaning": meaning,
        }
    )
pd.DataFrame(schema_rows)

,field,data_type,declared_role,meaning
0,unique_id,string,record identifier,Identifies a submitted row.
1,cluster,string,evaluation label,Known duplicate group; not used here.
2,full_name,string,derived identifier,Combined name representation.
3,first_and_surname,string,derived identifier,Combined name representation.
4,first_name,string,linkage column,Candidate personal identifier.
5,surname,string,linkage column,Candidate personal identifier.
6,dob,string,linkage column,Candidate date-of-birth identifier.
7,birth_place,string,linkage column,Candidate place identifier.
8,postcode_fake,string,linkage column,Synthetic postcode-like identifier.
9,gender,string,auxiliary variable,Available for subgroup assessment.


## 4. Dataset-level linkage overview

Start with the whole-dataset envelope before examining individual PII.
**All PII complete** describes rows available to strict exact rules.
**At least one PII usable** shows whether any declared identifier remains.
**Rows with a unique full signature** describes rows distinguished without
ties by the complete declared combination. **Maximum distinct 1:1
assignments** allows one assignment per usable signature and is therefore
a structural ceiling, not an expected match rate.

Record-ID checks concern the submitted row identifier. Exact duplicate
checks use `DUPLICATE_COLUMNS` and exclude the row ID and truth label.

In [4]:
dataset_overview_records = dataset_linkage_overview_data(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    record_id_column=RECORD_ID_COLUMN,
    duplicate_columns=DUPLICATE_COLUMNS,
    blank_as_null=BLANK_AS_NULL,
)
dataset_overview = pd.DataFrame(dataset_overview_records)
dataset_overview.style.format(
    {
        "complete_pii_share": "{:.1%}",
        "usable_pii_share": "{:.1%}",
        "unique_full_signature_share": "{:.1%}",
        "ambiguous_full_signature_share": "{:.1%}",
        "maximum_one_to_one_assignment_share": "{:.1%}",
        "exact_duplicate_record_share": "{:.1%}",
    }
)

,source_dataset,dataset_order,total_rows,linkage_columns,number_of_linkage_columns,complete_pii_share,usable_pii_share,unique_full_signature_share,ambiguous_full_signature_share,unusable_full_signature_share,maximum_one_to_one_assignment_share,largest_signature_group,exact_duplicate_record_share,exact_duplicate_records,largest_exact_duplicate_group,record_id_column,record_id_missing_rows,record_id_duplicate_records
0,Historical 50k,0,50578,first_name + surname + dob + postcode_fake + birth_place + occupation,6,32.2%,100.0%,71.3%,28.7%,0.000040,81.7%,8,26.1%,13215,8,unique_id,0,0


In [5]:
dataset_linkage_overview_chart(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    record_id_column=RECORD_ID_COLUMN,
    duplicate_columns=DUPLICATE_COLUMNS,
    blank_as_null=BLANK_AS_NULL,
)

## 5. Quality of individual linkage columns

- **Usable completeness:** how often a field contains something usable.
- **Distinct-value rate:** distinct values divided by observed rows.
- **Singleton-record share:** rows whose value occurs once.
- **Repeated-record share:** rows whose value is shared with other rows.
- **Largest-value share:** concentration in the most common value.

No single measure determines linkability. A field can be complete but
weakly distinguishing, or distinctive but frequently missing.

In [6]:
column_quality_records = linkage_column_quality_data(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
column_quality = pd.DataFrame(column_quality_records)
quality_view = column_quality[
    [
        "column_name",
        "total_rows",
        "observed_rows",
        "usable_completeness",
        "distinct_values",
        "distinct_value_rate",
        "singleton_record_share",
        "repeated_record_share",
        "largest_value_share",
        "top_5pct_values_record_share",
    ]
].copy()
quality_view.style.format(
    {
        "total_rows": "{:,.0f}",
        "observed_rows": "{:,.0f}",
        "usable_completeness": "{:.1%}",
        "distinct_values": "{:,.0f}",
        "distinct_value_rate": "{:.1%}",
        "singleton_record_share": "{:.1%}",
        "repeated_record_share": "{:.1%}",
        "largest_value_share": "{:.1%}",
        "top_5pct_values_record_share": "{:.1%}",
    }
)

,column_name,total_rows,observed_rows,usable_completeness,distinct_values,distinct_value_rate,singleton_record_share,repeated_record_share,largest_value_share,top_5pct_values_record_share
0,first_name,"50,578","50,511",99.9%,"4,413",8.7%,4.7%,95.3%,5.5%,73.6%
1,surname,"50,578","46,063",91.1%,"6,195",13.4%,4.9%,95.1%,1.3%,31.4%
2,dob,"50,578","39,175",77.5%,"8,985",22.9%,12.5%,87.5%,1.5%,43.0%
3,postcode_fake,"50,578","39,152",77.4%,"12,363",31.6%,16.3%,83.7%,0.1%,20.2%
4,birth_place,"50,578","43,651",86.3%,"2,373",5.4%,0.8%,99.2%,5.3%,45.2%
5,occupation,"50,578","25,289",50.0%,453,1.8%,0.2%,99.8%,12.1%,61.3%


In [7]:
linkage_column_quality_chart(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [8]:
lowest_completeness = column_quality.loc[column_quality["usable_completeness"].idxmin()]
highest_repetition = column_quality.loc[column_quality["repeated_record_share"].idxmax()]
highest_concentration = column_quality.loc[column_quality["largest_value_share"].idxmax()]

pd.DataFrame(
    [
        {
            "question": "Which assessed field is least complete?",
            "finding": lowest_completeness["column_name"],
            "evidence": f'{lowest_completeness["usable_completeness"]:.1%} usable',
            "linkage implication": "Missing values can prevent true pairs agreeing on this field.",
        },
        {
            "question": "Which field places most rows in repeated values?",
            "finding": highest_repetition["column_name"],
            "evidence": f'{highest_repetition["repeated_record_share"]:.1%} repeated',
            "linkage implication": "Other identifiers are needed to distinguish shared values.",
        },
        {
            "question": "Which field has the largest single-value concentration?",
            "finding": highest_concentration["column_name"],
            "evidence": f'{highest_concentration["largest_value_share"]:.1%} in its largest group',
            "linkage implication": "A common value can generate many candidates if used alone.",
        },
    ]
)

,question,finding,evidence,linkage implication
0,Which assessed field is least complete?,occupation,50.0% usable,Missing values can prevent true pairs agreeing...
1,Which field places most rows in repeated values?,occupation,99.8% repeated,Other identifiers are needed to distinguish sh...
2,Which field has the largest single-value conce...,occupation,12.1% in its largest group,A common value can generate many candidates if...


## 6. Missingness by PII and co-missingness pattern

Per-column missingness identifies which fields are unavailable.
Co-missingness identifies fields that are absent together. The complete
pattern table is ordered with the most common pattern first; the chart
displays the leading patterns for readability.

In [9]:
missingness_by_column_records = missingness_by_column_data(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
pd.DataFrame(missingness_by_column_records).style.format(
    {"missing_record_share": "{:.1%}"}
)

,source_dataset,column_name,column_order,total_rows,missing_rows,missing_record_share
0,Historical 50k,first_name,0,50578,67,0.1%
1,Historical 50k,surname,1,50578,4515,8.9%
2,Historical 50k,dob,2,50578,11403,22.5%
3,Historical 50k,postcode_fake,3,50578,11426,22.6%
4,Historical 50k,birth_place,4,50578,6927,13.7%
5,Historical 50k,occupation,5,50578,25289,50.0%


In [10]:
missingness_by_column_chart(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [11]:
missingness_patterns = missingness_pattern_data(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
pd.DataFrame(missingness_patterns).head(15).style.format(
    {"record_share": "{:.1%}"}
)

,source_dataset,missing_fields,present_fields,missing_field_count,records,record_share
0,Historical 50k,[none],first_name + surname + dob + postcode_fake + birth_place + occupation,0,16263,32.2%
1,Historical 50k,occupation,first_name + surname + dob + postcode_fake + birth_place,1,10877,21.5%
2,Historical 50k,dob + occupation,first_name + surname + postcode_fake + birth_place,2,3014,6.0%
3,Historical 50k,postcode_fake + occupation,first_name + surname + dob + birth_place,2,2985,5.9%
4,Historical 50k,postcode_fake,first_name + surname + dob + birth_place + occupation,1,2125,4.2%
5,Historical 50k,dob,first_name + surname + postcode_fake + birth_place + occupation,1,2079,4.1%
6,Historical 50k,dob + postcode_fake + occupation,first_name + surname + birth_place,3,1814,3.6%
7,Historical 50k,birth_place + occupation,first_name + surname + dob + postcode_fake,2,1483,2.9%
8,Historical 50k,birth_place,first_name + surname + dob + postcode_fake + occupation,1,1091,2.2%
9,Historical 50k,surname + occupation,first_name + dob + postcode_fake + birth_place,2,933,1.8%


In [12]:
missingness_pattern_chart(
    records,
    columns=LINKAGE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
    top_n=12,
)

In [13]:
pd.DataFrame(
    missingness_by_count_data(
        records,
        columns=LINKAGE_COLUMNS,
        table_names_for_chart=["Historical 50k"],
        blank_as_null=BLANK_AS_NULL,
    )
).style.format({"record_share": "{:.1%}"})

,source_dataset,missing_field_count,number_of_columns,records,record_share
0,Historical 50k,0,6,16263,32.2%
1,Historical 50k,1,6,17006,33.6%
2,Historical 50k,2,6,10761,21.3%
3,Historical 50k,3,6,5215,10.3%
4,Historical 50k,4,6,1213,2.4%
5,Historical 50k,5,6,118,0.2%
6,Historical 50k,6,6,2,0.0%


## 7. Cumulative uniqueness and value concentration

The cumulative curve ranks observed values from most to least frequent.
It shows how much of the dataset is occupied by a given share of distinct
values. The full dominance distribution then shows how many values and
records occur at each frequency. Observed identifier values are not
returned.

In [14]:
value_concentration_chart(
    records,
    columns=DISTRIBUTION_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [15]:
value_dominance_distribution_chart(
    records,
    columns=DISTRIBUTION_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

## 8. Combination ambiguity and relaxed exact keys

An exact signature is the selected field combination on a row. It is
**unique** if it occurs once and **ambiguous** if it occurs on several
rows. Removing fields shows what happens when an exact key is relaxed.

This is a within-dataset diagnostic, not a prediction of linkage accuracy.

In [16]:
ambiguity_records = combination_ambiguity_data(
    records,
    column_combinations=COLUMN_COMBINATIONS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
ambiguity = pd.DataFrame(ambiguity_records)
ambiguity_view = ambiguity[
    [
        "column_combination",
        "complete_record_share",
        "unique_record_share",
        "ambiguous_record_share",
        "unusable_record_share",
        "largest_signature_group",
        "maximum_one_to_one_assignment_share",
    ]
].copy()
ambiguity_view.style.format(
    {
        "complete_record_share": "{:.1%}",
        "unique_record_share": "{:.1%}",
        "ambiguous_record_share": "{:.1%}",
        "unusable_record_share": "{:.1%}",
        "largest_signature_group": "{:,.0f}",
        "maximum_one_to_one_assignment_share": "{:.1%}",
    }
)

,column_combination,complete_record_share,unique_record_share,ambiguous_record_share,unusable_record_share,largest_signature_group,maximum_one_to_one_assignment_share
0,first_name + surname + dob + postcode_fake,58.7%,62.5%,37.5%,0.0%,27,75.0%
1,surname + dob + postcode_fake,58.7%,48.6%,50.2%,1.2%,30,64.0%
2,first_name + dob + postcode_fake,62.9%,52.8%,47.2%,0.0%,149,66.7%
3,dob + postcode_fake,63.0%,34.2%,57.7%,8.1%,97,49.6%
4,surname + dob,71.5%,27.2%,69.9%,3.0%,96,43.2%
5,postcode_fake,77.4%,12.6%,64.8%,22.6%,34,24.4%


In [17]:
combination_ambiguity_chart(
    records,
    column_combinations=COLUMN_COMBINATIONS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

## 9. Combination cardinality and exact duplicate rows

Cardinality buckets show the number of rows sharing each exact linkage
signature. The duplicate assessment uses the declared submitted columns
exactly as stored and excludes the row identifier and evaluation label.

In [18]:
combination_size_distribution_chart(
    records,
    column_combinations=COLUMN_COMBINATIONS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [19]:
duplicate_records = exact_duplicate_data(
    records,
    columns=DUPLICATE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
)
pd.DataFrame(duplicate_records).style.format(
    {"duplicate_record_share": "{:.1%}"}
)

,source_dataset,columns_used,total_rows,duplicate_groups,records_in_duplicate_groups,duplicate_record_share,excess_duplicate_rows,largest_duplicate_group
0,Historical 50k,first_name + surname + dob + postcode_fake + birth_place + occupation + gender,50578,4930,13215,26.1%,8285,8


In [20]:
exact_duplicate_chart(
    records,
    columns=DUPLICATE_COLUMNS,
    table_names_for_chart=["Historical 50k"],
)

## 10. Full subgroup equity assessment

`AUXILIARY_COLUMNS` controls the stratification. The example uses gender;
add or replace columns when another approved grouping variable is more
relevant. This compares completeness patterns, not linkage outcomes.

Gaps are subgroup minus dataset average. The equity screen uses descriptive
triage thresholds of 2, 5 and 10 percentage points; these are not
statistical tests or evidence of realised linkage bias.

In [21]:
subgroup_missingness = subgroup_missingness_summary_data(
    records,
    columns=LINKAGE_COLUMNS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
pd.DataFrame(subgroup_missingness).style.format(
    {
        "any_missing_share": "{:.1%}",
        "most_common_incomplete_pattern_share": "{:.1%}",
    }
)

,source_dataset,auxiliary_column,auxiliary_value,group_records,any_missing_records,any_missing_share,most_common_incomplete_pattern,most_common_incomplete_pattern_share
0,Historical 50k,gender,male,32895,20345,61.8%,occupation,22.6%
1,Historical 50k,gender,[MISSING],11230,9688,86.3%,occupation,17.2%
2,Historical 50k,gender,female,6423,4265,66.4%,occupation,23.2%
3,Historical 50k,gender,t2227949966,12,7,58.3%,occupation,25.0%
4,Historical 50k,gender,t2227949951,10,5,50.0%,surname + occupation,20.0%
5,Historical 50k,gender,female | male,3,3,100.0%,occupation,100.0%
6,Historical 50k,gender,t2227949953,3,2,66.7%,surname + dob + postcode_fake + occupation,33.3%
7,Historical 50k,gender,t2227949974,2,0,0.0%,[none],0.0%


In [22]:
subgroup_missingness_chart(
    records,
    columns=LINKAGE_COLUMNS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [23]:
subgroup_fields = subgroup_linkage_column_quality_data(
    records,
    columns=LINKAGE_COLUMNS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
subgroup_field_view = pd.DataFrame(subgroup_fields)[
    [
        "auxiliary_column",
        "auxiliary_value",
        "column_name",
        "group_records",
        "usable_completeness",
        "usable_completeness_gap",
        "repeated_record_share",
        "repeated_record_share_gap",
        "top_5pct_values_record_share",
        "top_5pct_values_record_share_gap",
    ]
]
subgroup_field_view.style.format(
    {
        column: "{:+.1%}" if column.endswith("_gap") else "{:.1%}"
        for column in subgroup_field_view
        if "share" in column or "completeness" in column
    }
)

,auxiliary_column,auxiliary_value,column_name,group_records,usable_completeness,usable_completeness_gap,repeated_record_share,repeated_record_share_gap,top_5pct_values_record_share,top_5pct_values_record_share_gap
0,gender,t2227949974,first_name,2,100.0%,+0.1%,100.0%,+4.7%,100.0%,+26.4%
1,gender,t2227949966,first_name,12,100.0%,+0.1%,66.7%,-28.6%,66.7%,-6.9%
2,gender,female | male,first_name,3,100.0%,+0.1%,100.0%,+4.7%,100.0%,+26.4%
3,gender,t2227949953,first_name,3,100.0%,+0.1%,0.0%,-95.3%,33.3%,-40.3%
4,gender,t2227949951,first_name,10,90.0%,-9.9%,100.0%,+4.7%,55.6%,-18.0%
5,gender,male,first_name,32895,99.9%,+0.0%,95.2%,-0.1%,75.9%,+2.3%
6,gender,[MISSING],first_name,11230,99.8%,-0.1%,87.1%,-8.2%,60.2%,-13.4%
7,gender,female,first_name,6423,99.9%,+0.1%,90.3%,-4.9%,50.6%,-23.0%
8,gender,t2227949951,surname,10,50.0%,-41.1%,100.0%,+4.9%,100.0%,+68.6%
9,gender,t2227949974,surname,2,100.0%,+8.9%,100.0%,+4.9%,100.0%,+68.6%


In [24]:
subgroup_linkage_column_quality_chart(
    records,
    columns=LINKAGE_COLUMNS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [25]:
subgroup_combinations = subgroup_combination_ambiguity_data(
    records,
    column_combinations=SUBGROUP_COMBINATIONS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
pd.DataFrame(subgroup_combinations)[
    [
        "auxiliary_value",
        "column_combination",
        "group_records",
        "complete_record_share",
        "unique_record_share",
        "ambiguous_record_share",
        "maximum_one_to_one_assignment_share",
        "maximum_one_to_one_assignment_share_gap",
    ]
].style.format(
    {
        column: "{:+.1%}" if column.endswith("_gap") else "{:.1%}"
        for column in pd.DataFrame(subgroup_combinations)
        if "share" in column
    }
)

,auxiliary_value,column_combination,group_records,complete_record_share,unique_record_share,ambiguous_record_share,maximum_one_to_one_assignment_share,maximum_one_to_one_assignment_share_gap
0,t2227949974,first_name + surname + dob + postcode_fake,2,100.0%,0.0%,100.0%,50.0%,-25.0%
1,t2227949951,first_name + surname + dob + postcode_fake,10,50.0%,30.0%,60.0%,50.0%,-25.0%
2,male,first_name + surname + dob + postcode_fake,32895,65.4%,58.0%,42.0%,72.3%,-2.7%
3,female | male,first_name + surname + dob + postcode_fake,3,100.0%,33.3%,66.7%,66.7%,-8.3%
4,t2227949966,first_name + surname + dob + postcode_fake,12,66.7%,58.3%,41.7%,66.7%,-8.3%
5,t2227949953,first_name + surname + dob + postcode_fake,3,33.3%,100.0%,0.0%,100.0%,+25.0%
6,[MISSING],first_name + surname + dob + postcode_fake,11230,37.6%,94.2%,5.7%,96.9%,+21.9%
7,female,first_name + surname + dob + postcode_fake,6423,61.9%,66.8%,33.2%,78.2%,+3.2%
8,t2227949974,dob + postcode_fake,2,100.0%,0.0%,100.0%,50.0%,+0.4%
9,t2227949951,dob + postcode_fake,10,80.0%,20.0%,60.0%,30.0%,-19.6%


In [26]:
subgroup_combination_ambiguity_chart(
    records,
    column_combinations=SUBGROUP_COMBINATIONS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [27]:
subgroup_equity = subgroup_equity_gap_data(
    records,
    columns=LINKAGE_COLUMNS,
    column_combinations=SUBGROUP_COMBINATIONS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
pd.DataFrame(subgroup_equity).head(30).style.format(
    {
        "group_record_share": "{:.1%}",
        "largest_adverse_gap": "{:+.1%}",
        "adverse_gap_magnitude": "{:.1%}",
    }
)

,source_dataset,auxiliary_column,auxiliary_value,assessment_level,item,group_records,group_record_share,adverse_metric,largest_adverse_gap,adverse_gap_magnitude,flag_level
0,Historical 50k,gender,t2227949951,individual PII,postcode_fake,10,0.0%,top-5% concentration,-79.8%,79.8%,priority review
1,Historical 50k,gender,t2227949953,individual PII,postcode_fake,3,0.0%,top-5% concentration,-79.8%,79.8%,priority review
2,Historical 50k,gender,t2227949974,individual PII,postcode_fake,2,0.0%,top-5% concentration,-79.8%,79.8%,priority review
3,Historical 50k,gender,female | male,individual PII,surname,3,0.0%,top-5% concentration,-68.6%,68.6%,priority review
4,Historical 50k,gender,t2227949966,individual PII,surname,12,0.0%,top-5% concentration,-68.6%,68.6%,priority review
5,Historical 50k,gender,t2227949953,individual PII,surname,3,0.0%,top-5% concentration,-68.6%,68.6%,priority review
6,Historical 50k,gender,t2227949951,individual PII,surname,10,0.0%,top-5% concentration,-68.6%,68.6%,priority review
7,Historical 50k,gender,t2227949974,individual PII,surname,2,0.0%,top-5% concentration,-68.6%,68.6%,priority review
8,Historical 50k,gender,t2227949974,combined PII,first_name + surname + dob + postcode_fake,2,0.0%,ambiguous-signature share,-62.5%,62.5%,priority review
9,Historical 50k,gender,t2227949974,individual PII,dob,2,0.0%,top-5% concentration,-57.0%,57.0%,priority review


In [28]:
subgroup_equity_gap_chart(
    records,
    columns=LINKAGE_COLUMNS,
    column_combinations=SUBGROUP_COMBINATIONS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [29]:
subgroup_value_concentration_chart(
    records,
    columns=DISTRIBUTION_COLUMNS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

In [30]:
subgroup_value_dominance_distribution_chart(
    records,
    columns=DISTRIBUTION_COLUMNS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)

## 11. Reusable profile and optional reference comparison

The profile combines all aggregate sections in a serialisable dictionary.
A reference must be another profile produced with compatible column and
missingness declarations, such as a previous spine extract. Differences
are current minus reference and should not be interpreted as data-quality
change without considering population and coverage changes.

In [31]:
profile = build_pre_linkage_profile(
    records,
    columns=LINKAGE_COLUMNS,
    column_combinations=COLUMN_COMBINATIONS,
    auxiliary_columns=AUXILIARY_COLUMNS,
    duplicate_columns=DUPLICATE_COLUMNS,
    record_id_column=RECORD_ID_COLUMN,
    table_names_for_chart=["Historical 50k"],
    blank_as_null=BLANK_AS_NULL,
)
list(profile)

['profile_type',
 'columns',
 'column_combinations',
 'auxiliary_columns',
 'dataset_overview',
 'column_quality',
 'value_concentration',
 'value_dominance_distribution',
 'missingness_by_column',
 'missingness_by_count',
 'missingness_patterns',
 'combination_ambiguity',
 'combination_size_distribution',
 'exact_duplicates',
 'subgroup_missingness_patterns',
 'subgroup_missingness_summary',
 'subgroup_linkage_column_quality',
 'subgroup_combination_ambiguity',
 'subgroup_equity_gaps',
 'subgroup_value_concentration',
 'subgroup_value_dominance_distribution']

In [32]:
if REFERENCE_PROFILE_PATH:
    with Path(REFERENCE_PROFILE_PATH).open(encoding="utf-8") as handle:
        reference_profile = json.load(handle)
    display(
        pre_linkage_reference_comparison_chart(
            profile,
            reference_profile,
            current_label="Current submission",
            reference_label="Reference",
        )
    )
    display(
        reference_value_concentration_chart(
            profile,
            reference_profile,
            current_label="Current submission",
            reference_label="Reference",
        )
    )
    display(
        reference_value_dominance_distribution_chart(
            profile,
            reference_profile,
            current_label="Current submission",
            reference_label="Reference",
        )
    )
else:
    print("Set REFERENCE_PROFILE_PATH to compare with an aggregate reference.")

Set REFERENCE_PROFILE_PATH to compare with an aggregate reference.


## 12. What this means for the next linkage stage

1. Avoid relying on a field that is frequently missing.
2. Combine common fields with more distinguishing identifiers.
3. Test relaxed combinations with Splink's blocking analysis before scoring.
4. Assess subgroup differences where appropriate auxiliary variables exist.
5. Evaluate the fitted linkage model separately using appropriate evidence.

These are structural opportunities and constraints. They are not pass/fail
rules and do not establish precision or recall.

## 13. Optional export

Set `EXPORT_OUTPUTS=True` below to save the reusable aggregate profile,
selected CSV files and standalone Splink charts. The existing `prelinkage_assessment` project
still contains the fuller composite HTML report. It has not been deleted
or replaced by this notebook; only the generic core is being redeveloped
inside Splink at this stage.

In [33]:
EXPORT_OUTPUTS = False
OUTPUT_DIR = project_root / "example_outputs" / "historical_50k_pre_linkage"

if EXPORT_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    dataset_overview.to_csv(
        OUTPUT_DIR / "dataset_linkage_overview.csv", index=False
    )
    column_quality.to_csv(OUTPUT_DIR / "linkage_column_quality.csv", index=False)
    ambiguity.to_csv(OUTPUT_DIR / "combination_ambiguity.csv", index=False)
    pd.DataFrame(missingness_by_column_records).to_csv(
        OUTPUT_DIR / "missingness_by_column.csv", index=False
    )
    pd.DataFrame(missingness_patterns).to_csv(
        OUTPUT_DIR / "missingness_patterns.csv", index=False
    )
    pd.DataFrame(subgroup_missingness).to_csv(
        OUTPUT_DIR / "subgroup_missingness.csv", index=False
    )
    pd.DataFrame(duplicate_records).to_csv(OUTPUT_DIR / "exact_duplicates.csv", index=False)
    with (OUTPUT_DIR / "pre_linkage_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "dataset": "Splink historical 50k",
                "records": historical_50k.num_rows,
                "linkage_column_quality": column_quality_records,
                "combination_ambiguity": ambiguity_records,
            },
            handle,
            indent=2,
        )
    with (OUTPUT_DIR / "pre_linkage_profile.json").open("w", encoding="utf-8") as handle:
        json.dump(profile, handle, indent=2)
    linkage_column_quality_chart(
        records,
        columns=LINKAGE_COLUMNS,
        table_names_for_chart=["Historical 50k"],
    ).save_offline_chart(
        OUTPUT_DIR / "linkage_column_quality.html",
        overwrite=True,
        print_msg=False,
    )
    combination_ambiguity_chart(
        records,
        column_combinations=COLUMN_COMBINATIONS,
        table_names_for_chart=["Historical 50k"],
    ).save_offline_chart(
        OUTPUT_DIR / "combination_ambiguity.html",
        overwrite=True,
        print_msg=False,
    )
    print(f"Outputs written to {OUTPUT_DIR}")
else:
    print("Nothing exported. Set EXPORT_OUTPUTS=True to save outputs.")

Nothing exported. Set EXPORT_OUTPUTS=True to save outputs.
